# G1 humanoid — cost-mapping check + normal / FPL / FPL+soft-min comparison

The Unitree **G1** (`g1_standup` task, ~36 DOF). It starts from the `stand` keyframe, so the task is really **keep standing / recover balance**. We (1) check whether the normal-cost → FPL `[0,1]` mapping is faithful, then (2) play a side-by-side video comparing vanilla MPPI, FPL, and FPL + soft-min-over-time on a **shove-recovery** disturbance.

**Cost-mapping audit (fixed in `analytic_mppi/tasks/g1_standup.py` this session):**

| objective | normal cost | FPL fulfillment `[0,1]` | issue found → fix |
|---|---|---|---|
| orientation | `10·(1−rz)` | `clip((rz−0.5)/0.45, 0, 1)` | **normal was `10·Σorient²` ≡ 10 (constant!)** — `orient` is a rotated *unit* vector, so it never influenced MPPI. Fixed to the tilt `1−rz`. FPL atom `(rz+1)/2` decayed too late (0.5 at horizontal) → reshaped to decay before horizontal. |
| height | `10·(h−0.9)²` | `1 − |h−0.9|/0.9` (clipped) | faithful (both target 0.9). Minor: the robot actually stands at ~0.98, so standing scores ~0.91 on height. |
| nominal (posture) | `0.1·Σ(qⱼ−q*ⱼ)²` | geom-mean of `1−|Δqⱼ|/dev` | faithful; a bit saturated high (~0.8–0.98). |

**Takeaway from the comparison (below):** on humanoid balance the three objectives **cooperate** (standing satisfies upright + height + posture at once), so FPL's weakest-link conjunction adds no advantage over a well-shaped quadratic cost — unlike the hopper, where speed vs stability genuinely compete. This matches FPL theory: the structural win needs *competing* objectives.

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import mujoco
from analytic_mppi.tasks import make_task
from analytic_mppi.tasks.g1_standup import _quat_rotate
from analytic_mppi.dynamics import MujocoBackend
from analytic_mppi.eval import render_video, run_episode
from analytic_mppi.controllers import MPPIv2

TASK = "g1_standup"
task = make_task(TASK)
DT = MujocoBackend(task.model_path).dt
print(f"task={TASK}  nq={task.nq} nv={task.nv} nu={task.nu} dt={DT}")

def init_stand(backend):
    """Start from the 'stand' keyframe at rest."""
    kf = backend.model.keyframe("stand")
    backend.data.qpos[:] = kf.qpos
    backend.data.qvel[:] = 0.0
    mujoco.mj_forward(backend.model, backend.data)

def make_shove(vx):
    """Start standing, then apply an initial forward torso velocity (a 'shove')."""
    def _init(backend):
        init_stand(backend)
        backend.data.qvel[0] = vx      # free-joint forward linear velocity
        mujoco.mj_forward(backend.model, backend.data)
    return _init

## 1. Cost-mapping check

Confirm the normal orientation cost is no longer a constant, and measure each FPL fulfillment's distribution over a batch of sampled rollouts from a standing start — a "live" atom spans a useful range; a dead one is pinned at 0 or 1.

In [ ]:
backend = MujocoBackend(task.model_path)
init_stand(backend)
s0 = backend.get_state()
sd0 = np.asarray(backend.data.sensordata, dtype=np.float64)
print(f"standing: torso_height={task._torso_height(sd0):.3f}  orient_rz={task._torso_orientation(sd0)[...,2]:.3f}")

# --- batch of noisy rollouts from the stand ---
K, H = 200, 25
rng = np.random.default_rng(0)
init = np.broadcast_to(s0, (K, backend.nstate)).copy()
mid = 0.5 * (task.u_min + task.u_max); half = 0.5 * (task.u_max - task.u_min)
ctrls = mid[None, None, :] + rng.uniform(-1, 1, (K, H, task.nu)) * half[None, None, :] * 0.5
states, sd = backend.rollout(init, ctrls)
qpos = task.qpos_of(states)

# (1) normal orientation cost is no longer constant
oc = task.running_cost_terms(qpos, None, sd, ctrls)[..., 0]   # orientation_cost column
print(f"normal orientation_cost: min={oc.min():.3f} max={oc.max():.3f}  "
      f"({'CONSTANT (bug!)' if np.allclose(oc, oc.flat[0]) else 'varies — fixed'})")

# (2) FPL atom distributions
atoms = {"orientation": task._orientation_fulfillment(sd),
         "height     ": task._height_fulfillment(sd),
         "nominal    ": task._nominal_fulfillment(qpos)}
print(f"\n{'atom':12s}  p05    med    p95    frac~1  frac~0")
for nm, a in atoms.items():
    a = a.ravel(); q = np.percentile(a, [5, 50, 95])
    print(f"{nm}  {q[0]:.3f}  {q[1]:.3f}  {q[2]:.3f}   {np.mean(a>0.99):.2f}    {np.mean(a<0.01):.2f}")
F = np.stack(list(atoms.values()), -1); amin = F.argmin(-1).ravel()
print("binding (min) atom: " + "  ".join(f"{n.strip()}={100*np.mean(amin==i):.0f}%"
                                          for i, n in enumerate(atoms)))

## 2. Video comparison — shove recovery

Give the standing humanoid a forward **shove** (initial torso velocity) and watch each controller recover. Captions show final root height and min uprightness (`rz`, 1 = upright).

- **normal MPPI** — vanilla MPPI on the (now-fixed) quadratic cost.
- **fpl (p=−2)** — FPL weakest-link conjunction over the same objectives, time-average aggregation.
- **fpl (p=−2) + soft-min** — FPL judged by its *worst moment* over time (`fpl_time_p=−2`).

Expect all three to keep the torso upright but settle into a **crouch**; the well-shaped normal cost tends to recover height best. The objectives here cooperate, so this is the "FPL ≈ baseline" regime (contrast with `hopper.ipynb`).

In [ ]:
import base64
from IPython.display import HTML, display

STEPS_CMP = 150
SEED_CMP  = 0
SHOVE     = 4.0           # m/s forward torso velocity at t=0
SHARED    = dict(num_samples=128, plan_horizon=0.5, num_knots=4, spline_type="zero", noise_level=0.3)

# (label, cost_mode, fpl_p, temperature, fpl_time_p)
RUNS = [
    ("normal MPPI",              "normal",   None, 1.0, None),
    ("fpl (p=-2)",               "fpl_cost", -2.0, 0.1, None),
    ("fpl (p=-2) + soft-min",    "fpl_cost", -2.0, 0.1, -2.0),
]

def _render(tag, cost_mode, fpl_p, temperature, fpl_time_p):
    kw = dict(cost_mode=cost_mode, temperature=temperature)
    if fpl_p is not None:      kw["fpl_p"] = fpl_p
    if fpl_time_p is not None: kw["fpl_time_p"] = fpl_time_p
    return render_video(TASK, MPPIv2, steps=STEPS_CMP, out_path=f"/tmp/g1_{tag}.mp4",
                        init_fn=make_shove(SHOVE), seed=SEED_CMP, width=420, height=420,
                        **SHARED, **kw)

def _stats(res):
    sd = res["sd"]                                    # torso sensors (standing height ~0.98)
    h = task._torso_height(sd)                        # torso height
    rz = task._torso_orientation(sd)[..., 2]          # torso up-vector z (1 = upright)
    return float(h[-1]), float(rz[-1]), float(rz.min())

def _vid(path, label, width=300):
    b64 = base64.b64encode(open(path, "rb").read()).decode()
    return (f'<figure style="display:inline-block;margin:6px;text-align:center;vertical-align:top">'
            f'<figcaption style="font-family:monospace;font-size:11px">{label}</figcaption>'
            f'<video width="{width}" controls autoplay loop muted>'
            f'<source src="data:video/mp4;base64,{b64}" type="video/mp4"></video></figure>')

print(f"shove={SHOVE} m/s, {STEPS_CMP} steps, seed {SEED_CMP}  (standing torso height ~0.98)")
html = ""
for i, (label, cm, p, t, tp) in enumerate(RUNS):
    res = _render(f"cmp{i}", cm, p, t, tp)
    hend, rzend, rzmin = _stats(res)
    print(f"{label:26s} height_end={hend:.2f}  rz_end={rzend:.2f}  rz_min={rzmin:.2f}  ({res['plan_ms']:.0f} ms/step)")
    html += _vid(res["path"], f"{label}<br>height {hend:.2f}, rz_end {rzend:.2f}, rz_min {rzmin:.2f}")
display(HTML(html))